# 11.6 先對齊再問答，怎麼和直接問答公平比較？


一個方案先教圖片描述，再教問答；另一個直接用問答學接頭與回應。如果第一方案多花一整段訓練，最後較好可能只是多用了資料。先把兩者的總有效回答位置排成相同預算。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
budget = 10000
plans = {
    "兩階段": {"描述對齊": 4000, "問答微調": 6000},
    "直接問答": {"描述對齊": 0, "問答微調": 10000},
}
for name, plan in plans.items():
    total = sum(plan.values())
    print(name, plan, "總有效位置", total, "符合預算", total == budget)

兩階段分 4,000＋6,000，直接問答用 10,000，程式都印總數 10,000、True。有效位置指真正計入答案代價的位置，不是問題字數，也不是完整樣本數；長答與短答混合時，相同步數未必同量。

這張表只是預算核對。實際比較還要同一初始模型、明確的更新範圍、同一批留出圖文題，以及基模原本會做的文字題。第一階段接頭需要帶到第二階段，不能花完預算又從隨機接頭重新開始。

既有小對照將兩階段合計 18,238 個有效目標，直接方案到最後一批為 18,256，差 18 個。最後圖片問答分別 9/12、10/12；一題差距不支持普遍優劣，也不能說前段對齊完全沒影響。直接版本未另列原文字留出分數，所以這個比較不支持文字保留結論。

描述對齊與問答 SFT 都可以用文字預測代價，差在材料、任務與開放參數。兩種是可比較安排，不是規定每個成品都必須先後做完。

練習把對齊 4,000 改 2,000，問答要改 8,000 才保持總量。只改第一項，預算核對會 False；計畫改了，測量結果不能跟著手改。

<details>
<summary>回顧與查證</summary>

可回顧：[11.3描述對齊](https://birdhackor.github.io/tiny-perceptron-vlm/11.3.html)、[11.4圖片問答](https://birdhackor.github.io/tiny-perceptron-vlm/11.4.html)、[7.3有效學習位置](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/vqa.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
